In [ ]:
import pandas, copy

from tqdm import tqdm

pandas.options.display.max_columns=999

In [ ]:
cog=pandas.read_csv('../create-coguk-dataset/cog_dataset-20210624.csv')

# Exploring the dataset

In [ ]:
cog[:3]

In [ ]:
cog=cog.loc[cog.object_store_bucket.notna()]

In [ ]:
print("This gives us "+str(len(cog))+" COGUK samples with an ENA accession number and FASTQ files in the ena bucket in object store")

print("There are " +str(len(cog.loc[cog.lineage.isna()]))+" samples with no PANGO lineage and " +str(len(cog.loc[cog.scorpio_call.isna()]))+ " samples with no SCORPIO lineage")

Let's look at the `pango` lineages and `scorpio` constellations

In [ ]:
cog.lineage.value_counts(dropna=False)

In [ ]:
cog.scorpio_call.value_counts(dropna=False)

What about the P.1 lineage?

In [ ]:
cog.loc[cog.lineage.str[0]=='P'].scorpio_call.value_counts()

In [ ]:
cog.loc[cog.lineage.str[0]=='P'].lineage.value_counts()

In [ ]:
cog.loc[cog.lineage=='None'].instrument_platform.value_counts()

And the newer Iota and Epsilon VUIs?

In [ ]:
cog.loc[cog.scorpio_call=='Iota (B.1.526-like)'].lineage.value_counts()

In [ ]:
cog.loc[cog.scorpio_call=='Epsilon (B.1.427/429-like)'].lineage.value_counts()

# Logic for creating testset

Let's arbitrarily choose the top 42 lineages, then add the others we want to sample and pick two samples from each.

I've picked 48 since that will fit onto a single `VM.Standard2.24` compute shape in OCI

This is all very arbitrary and can be changed!

In [ ]:
a=cog.lineage.value_counts()
lineages_for_testset={}
lineages=list(a.loc[a>100][:50].index)
for i in lineages:
    lineages_for_testset[i]=4

lineages_for_testset['P.1']=1
lineages_for_testset['P.2']=4
lineages_for_testset    

lineages_for_testset['B.1.617.2']=2
lineages_for_testset['B.1.287']=1
lineages_for_testset['B.1.36.25']=1
lineages_for_testset['B.1.400']=1
lineages_for_testset['None']=4
# print(lineages_for_testset)
# len(lineages_for_testset)

samples=0
for i in lineages_for_testset:
    samples+=lineages_for_testset[i] 
print(samples)

In [ ]:
len(lineages_for_testset)

## Algorithm

1. for a given lineage, randomly pick 10 samples (with replacement to deal with cases where there are fewer than 10 samples in that lineage)
2. for each, make a set out of its mutations ($A_i$)
3. for the samples stored to date, make a single set of all their collective mutations ($B$)
4. by considering $B-A_i$ in turn, choose the sample which has the most mutations not already in $B$
5. add to to the testset and repeat

In [ ]:
def create_diverse_testset(coguk_df,required_lineages,random_seed=42,samples_to_pick=50):

    testset={}
    
    number_samples=0

    # iterate through the list of pango lineages we wish to cover in the testset
    for lineage in tqdm(lineages_for_testset):

        # before we pick some new samples to consider, build the set of existing mutations
        existing_mutations=set()    
        for idx in testset:    
            existing_mutations|=testset[idx]['mutations']

        # now 
        df=copy.deepcopy(coguk_df.loc[(coguk_df.lineage==lineage)])

        for n in range(lineages_for_testset[lineage]):
        
            try:
                random_samples=list(df.sample(n=samples_to_pick,random_state=random_seed,replace=True).index)
            except:
                print("No samples available for lineage "+lineage)
                continue 
            random_seed+=1

            sample_set={}
            additional_mutations=0
            next_sample=None

            for idx in random_samples:

                del1=cog.loc[idx].del_1605_3
                del2=cog.loc[idx].del_21765_6

                mutations=set()

                if del1=='X':
                    mutations|=set(['del_1605_3_X'])
                elif del1=='del':
                    mutations|=set(['del_1605_3'])

                if del2=='X':
                    mutations|=set(['del_21765_3_X'])
                elif del2=='del':
                    mutations|=set(['del_21765_3'])

                sample_set[idx]={}
                if isinstance(cog.loc[idx].mutations,str):
                    mutations|=set(cog.loc[idx].mutations.split('|'))      

                sample_set[idx]['mutations']=mutations
                sample_set[idx]['pango_lineage']=cog.loc[idx].lineage
                sample_set[idx]['scorpio_lineage']=cog.loc[idx].scorpio_call

            next_sample=random_samples[0]

            for idx in random_samples:    
                if idx in sample_set.keys() and len(sample_set[idx]['mutations']-existing_mutations)> additional_mutations:
                    next_sample=idx
                    additional_mutations=len(sample_set[idx]['mutations']-existing_mutations)
            try:
                testset[next_sample]=sample_set[next_sample]                
            except:
                print(next_sample,sample_set)

    muts=set()
    for i in testset:
        muts|=testset[i]['mutations']
        
    print("There are "+str(len(testset))+" samples in the testset with "+str(len(muts))+" unique mutations")
    
    return(testset.keys())




In [ ]:
df=copy.deepcopy(cog.loc[(cog.instrument_platform=='OXFORD_NANOPORE')])

samples=create_diverse_testset(df,lineages_for_testset)

nanopore_testset=cog.loc[samples]

nanopore_testset.to_csv('cog-testset-genetically-diverse-'+str(len(nanopore_testset))+'samples-nanopore-v0.1.csv.gz')

nanopore_testset[['object_store_bucket','run_accession']].to_csv('sp3-testset-genetically-diverse-'+str(len(nanopore_testset))+'samples-nanopore-v0.1.csv',index=False,header=False)

In [ ]:
len(samples)

In [ ]:
df=copy.deepcopy(cog.loc[(cog.instrument_platform=='ILLUMINA')])

samples=create_diverse_testset(df,lineages_for_testset)

illumina_testset=cog.loc[samples]

illumina_testset.to_csv('cog-testset-genetically-diverse-'+str(len(illumina_testset))+'samples-illumina-v0.1.csv.gz')

illumina_testset[['object_store_bucket','run_accession']].to_csv('sp3-testset-genetically-diverse-'+str(len(illumina_testset))+'samples-illumina-v0.1.csv',\
                                                    index=False,header=False)

In [ ]:
illumina_testset=pandas.read_csv('cog-testset-genetically-diverse-205samples-illumina-v0.1.csv.gz')

In [ ]:
foo=pandas.read_csv('cog-testset-genetically-diverse-48samples-illumina-v0.1.csv.gz')

In [ ]:
duplicates=set(foo.cogid).intersection(set(illumina_testset.cogid))

In [ ]:
illumina_testset=pandas.concat([foo[~foo.cogid.isin(duplicates)],illumina_testset])

In [ ]:
illumina_testset